## Airline AI Assistant


In [1]:
# Imports: `gradio` for building the chat UI and `OpenAI` for model client usage

import gradio as gr
from openai import OpenAI
import sqlite3

d:\CodeBase\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Connect to the SQLite database (or create it if it doesn't exist)
db = sqlite3.connect('flight.db')

# Define the table schema and create the table if it doesn't exist
cursor = db.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS flight(
        flight_id TEXT PRIMARY KEY,
        origin_city TEXT,
        destination_city TEXT,
        time TEXT,
        price REAL,
        airline TEXT,
        duration TEXT
        )'''
    )

db.commit()


In [3]:
import json

with open('flights_data.json', 'r') as f:
    json_data = json.load(f)

# Insert data into the database
for flight in json_data["flights"]:
    cursor.execute('''
        INSERT OR IGNORE INTO flight (flight_id, origin_city, destination_city, time, price, airline, duration)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (
        flight['flight_id'],
        flight['origin_city'],
        flight['destination_city'],
        flight['time'],
        flight['price'],
        flight['airline'],
        flight['duration']
    ))

db.commit()

In [7]:
# Quick health-check: verify the Ollama server is reachable before making requests
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [8]:
# Start the Ollama server (shell magic).
!ollama serve

Error: listen tcp 127.0.0.1:11434: bind: Only one usage of each socket address (protocol/network address/port) is normally permitted.


In [9]:
# Initialize the Ollama/OpenAI client pointing at the local server

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [10]:
# System prompt: defines assistant behavior, tone, and response constraints

system_prompt = """
You are a helpful customer service assistant for FlightAI airline.
Keep responses brief and courteous (1-2 sentences).
Use the provided tools to fetch flight information when needed.
If a flight doesn't exist, inform the customer clearly.
Always be accurate. If unsure, say so.
"""

## Tools
### Define the tool functions to interact with the SQLite database and fetch the required information.
- `get_flight_details(origin, destination)`: Fetches flight details for a specific route from the database.
- `get_flights_from_origin(origin)`: Fetches all flights departing from a specific origin city from the database.
- `get_flights_to_destination(destination)`: Fetches all flights arriving at a specific destination from the database.
- `flights_available_for_airline(airline)`: Fetches all flights operated by a specific airline from the database.
- `get_flight_price(origin, destination)`: Fetches the price for a flight on a specific route from the database.
- `get_flight_time(origin, destination)`: Fetches the departure time for a flight on a specific route from the database.
- `get_flight_duration(origin, destination)`: Fetches the duration for a flight on a specific route from the database.

In [11]:
# Step 1: Define tool functions to interact with the SQLite database and fetch the required information.

def get_flight_details(origin, destination):
    print(f"tool called: get_flight_details({origin}, {destination})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM prices WHERE origin_city = ? and destination_city = ? COLLATE NOCASE', (origin.lower(),destination.lower()))
        result = cursor.fetchone()
    if result:     # result is a tuple with the flight details
        return f"Flight details for {origin} to {destination}: Flight Name - {result[0]}, Time - {result[3]}, Price - ₹{result[4]}, Airline - {result[5]}, Duration - {result[6]}."
    else:
        return f"No flight found from {origin} to {destination}."
    
def get_flights_from_origin(origin):
    print(f"tool called: get_flights_from_origin({origin})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT destination_city, flight_id, time, price, airline FROM flight WHERE origin_city = ? COLLATE NOCASE', (origin.lower(),))
        flights = cursor.fetchall()
    flight_info = [f"Flight details for {origin}: Destination - {flight[0]}, Flight Name - {flight[1]}, Time - {flight[2]}, Price - ₹{flight[3]}, Airline - {flight[4]}." for flight in flights]
    if flight_info:
        return "\n".join(flight_info)
    return f"No flight found from {origin}."

def get_flights_to_destination(destination):
    print(f"tool called: get_flights_to_destination({destination})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT origin_city, flight_id, time, price, airline FROM flight WHERE destination_city = ? COLLATE NOCASE', (destination.lower(),))
        flights = cursor.fetchall()
    flight_info = [f"Flight details for {destination}: Origin - {flight[0]}, Flight Name - {flight[1]}, Time - {flight[2]}, Price - ₹{flight[3]}, Airline - {flight[4]}." for flight in flights]
    if flight_info:
        return "\n".join(flight_info)
    return f"No flight found to {destination}."

def flights_available_for_airline(airline):
    print(f"tool called: flights_available_for_airline({airline})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT origin_city, destination_city, flight_id, time FROM flight WHERE airline = ? COLLATE NOCASE', (airline.lower(),))
        flights = cursor.fetchall()
    flight_info = [f"Flight details: Origin - {flight[0]}, Destination - {flight[1]}, Flight Name - {flight[2]}, Time - {flight[3]}." for flight in flights]
    if flight_info:
        return "\n".join(flight_info)
    return "No flights available for the specified airline."

def get_flight_price(origin, destination):
    print(f"tool called: get_flight_price({origin}, {destination})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM flight WHERE origin_city = ? and destination_city = ? COLLATE NOCASE', (origin.lower(),destination.lower()))
        result = cursor.fetchone()
    if result:
        return f"The price for a flight from {origin} to {destination} is ₹{result[0]}."
    else:
        return f"No flight found from {origin} to {destination}."

def get_flight_time(origin, destination):
    print(f"tool called: get_flight_time({origin}, {destination})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT time FROM flight WHERE origin_city = ? and destination_city = ? COLLATE NOCASE', (origin.lower(),destination.lower()))
        result = cursor.fetchone()
    if result:
        return f"The flight time from {origin} to {destination} is {result[0]}."
    else:
        return f"No flight found from {origin} to {destination}."
    
def get_flight_duration(origin, destination):
    print(f"tool called: get_flight_duration({origin}, {destination})")
    with sqlite3.connect("flight.db") as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT duration FROM flight WHERE Lower(origin_city) = ? and Lower(destination_city) = ?', (origin.lower(),destination.lower()))
        result = cursor.fetchone()
    if result:
        return f"The flight duration from {origin} to {destination} is {result[0]}."
    else:
        return f"No flight found from {origin} to {destination}."

## Tools Metadata
### Define metadata for the tools to help the assistant understand when and how to use them.


In [12]:
# Step 2: Tool metadata used for function-calling (e.g., for an LLM that supports calling typed tools)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_flight_details",
            "description": "Fetches flight details for a specific route from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flights_from_origin",
            "description": "Fetches all flights departing from a specific origin city from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"}
                },
                "required": ["origin"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flights_to_destination",
            "description": "Fetches all flights arriving at a specific destination from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "flights_available_for_airline",
            "description": "Fetches all flights operated by a specific airline from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "airline": {"type": "string", "description": "Airline name"}
                },
                "required": ["airline"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_price",
            "description": "Fetches the price for a flight on a specific route from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_time",
            "description": "Fetches the departure time for a flight on a specific route from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_duration",
            "description": "Fetches the duration for a flight on a specific route from the database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {"type": "string", "description": "Origin city"},
                    "destination": {"type": "string", "description": "Destination city"}
                },
                "required": ["origin", "destination"]
            }
        }
    }
]

In [13]:
# Step 3: Map tool names to the actual Python function objects so the assistant can call them

tool_functions = {
    "get_flight_details": get_flight_details,
    "get_flights_from_origin": get_flights_from_origin,
    "get_flights_to_destination": get_flights_to_destination,
    "flights_available_for_airline": flights_available_for_airline,
    "get_flight_price": get_flight_price,
    "get_flight_time": get_flight_time,
    "get_flight_duration": get_flight_duration
}

In [14]:
# Step 4: Handle tool calls from the model response (OpenAI/Ollama style)
import json

def handle_tool_calls(message):
    """
    Handle all tool calls from a model message and return a list of tool response messages.
    """
    if not message.tool_calls:
        return []

    tool_responses = []

    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
        except json.JSONDecodeError:
            arguments = {}

        if tool_name in tool_functions:
            try:
                tool_result = tool_functions[tool_name](**arguments)
                content = str(tool_result)
            except Exception as e:
                content = f"Error executing {tool_name}: {str(e)}"
        else:
            content = f"Tool {tool_name} not found."

        tool_responses.append({
            "role": "tool",
            "name": tool_name,
            "content": content,
            "tool_call_id": tool_call.id
        })
    
    print(f"Generated tool responses: {tool_responses}")
    return tool_responses

In [15]:
# Step 5: Enhanced chat function with tool-calling support
# This version uses a single model for both generation and tool response processing

def chat_with_tools(user_message, chat_history):
    """
    Enhanced chat function that supports tool calling using a single model.
    The model can decide to call tools to gather flight information before responding.
    """
    model_name = "qwen2.5:7b-instruct"
    # model_name = "qwen3.5:4b-q4_K_M"      # thinking model
    # model_name = "llama3.2"

    messages = (
        [{"role": "system", "content": system_prompt}]
        + chat_history
        + [{"role": "user", "content": user_message}]
    )

    while True:
        response = ollama.chat.completions.create(
            model=model_name,
            messages=messages,
            extra_body={"think": False},  # Disable internal "thinking" to get immediate tool calls
            tools=tools,
        )

        assistant_message = response.choices[0].message  # message = {"role": "assistant", "content": "...", "tool_calls": [...]}
        messages.append(assistant_message)
        
        if assistant_message.tool_calls:
            tool_responses = handle_tool_calls(assistant_message)
            messages.extend(tool_responses)
            # Loop continues to let the model process tool results and provide a final answer
        else:
            # No more tool calls, return final content
            return assistant_message.content

In [ ]:
# Launch the Gradio Chat UI with the `chat_with_tools` function

gr.ChatInterface(fn=chat_with_tools, title="Airline AI Assistant").launch(show_error=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


tool called: get_flights_from_origin(delhi)
tool called: get_flights_to_destination(mumbai)
Generated tool responses: [{'role': 'tool', 'name': 'get_flights_from_origin', 'content': 'Flight details for delhi: Destination - Mumbai, Flight Name - FL-001, Time - 08:00, Price - ₹5500.0, Airline - AirIndia.\nFlight details for delhi: Destination - Kolkata, Flight Name - FL-003, Time - 12:15, Price - ₹6200.0, Airline - GoAir.\nFlight details for delhi: Destination - Hyderabad, Flight Name - FL-022, Time - 06:00, Price - ₹5300.0, Airline - IndiGo.\nFlight details for delhi: Destination - Pune, Flight Name - FL-027, Time - 07:30, Price - ₹5800.0, Airline - IndiGo.', 'tool_call_id': 'call_u0pyaeaw'}, {'role': 'tool', 'name': 'get_flights_to_destination', 'content': 'Flight details for mumbai: Origin - Delhi, Flight Name - FL-001, Time - 08:00, Price - ₹5500.0, Airline - AirIndia.\nFlight details for mumbai: Origin - Kolkata, Flight Name - FL-007, Time - 11:30, Price - ₹6800.0, Airline - SpiceJe